# Chapitre 2 — Optimisation et dynamique d’entraînement

Parcours corrigé : NumPy + Matplotlib ; cellule PyTorch finale optionnelle.
Prédire → tester → expliquer → transférer.


## Étape 1
Préparer NumPy/Matplotlib et consigner les versions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
print('NumPy :', np.__version__)



## Étape 2
Compléter les états et expliquer les conventions.


In [ ]:
# CELL 2 — Une mise à jour, un état indépendant par expérience.
class Optimizer:
    def __init__(self, params, kind='sgd', lr=.01, momentum=.9,
                 beta1=.9, beta2=.999, eps=1e-8, wd=0.):
        self.kind, self.lr, self.mu = kind, lr, momentum
        self.b1, self.b2, self.eps, self.wd = beta1, beta2, eps, wd
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        for k, p in params.items():
            g = grads[k]
            if self.kind == 'sgd':
                direction = g
            elif self.kind == 'momentum':
                self.m[k] = self.mu * self.m[k] + g
                direction = self.m[k]
            elif self.kind == 'rmsprop':
                self.v[k] = self.b2 * self.v[k] + (1-self.b2) * g*g
                direction = g / (np.sqrt(self.v[k]) + self.eps)
            elif self.kind in ('adam', 'adamw'):
                self.m[k] = self.b1 * self.m[k] + (1-self.b1) * g
                self.v[k] = self.b2 * self.v[k] + (1-self.b2) * g*g
                mh = self.m[k] / (1-self.b1**self.t)
                vh = self.v[k] / (1-self.b2**self.t)
                direction = mh / (np.sqrt(vh) + self.eps)
            else:
                raise ValueError(self.kind)
            # Décroissance découplée, uniquement sur les matrices de poids.
            if self.kind == 'adamw' and k.startswith('W'):
                p *= 1-self.lr*self.wd
            p -= self.lr * direction



## Étape 3
Prédire la stabilité avant les courbes.


In [ ]:
# CELL 3 — Géométrie quadratique : expérience déterministe.
A = np.diag([1., 40.])
def qloss(w): return .5 * w @ A @ w
def trajectory(kind, lr, steps=100):
    p = {'W': np.array([3., 1.])}
    opt = Optimizer(p, kind=kind, lr=lr)
    path = [p['W'].copy()]
    for _ in range(steps):
        opt.step(p, {'W': A @ p['W']})
        path.append(p['W'].copy())
    return np.array(path)

paths = {name: trajectory(name, lr) for name, lr in
         [('sgd', .04), ('momentum', .02), ('adam', .1)]}
assert qloss(paths['sgd'][-1]) < qloss(paths['sgd'][0])
bad = trajectory('sgd', .06, steps=20)
assert qloss(bad[-1]) > qloss(bad[0])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for name, path in paths.items():
    ax[0].plot(path[:, 0], path[:, 1], '.-', markersize=2, label=name)
    ax[1].semilogy([qloss(w) for w in path], label=name)
ax[0].set(xlabel='w1', ylabel='w2', title='Trajectoires ; pas spécifiques')
ax[1].set(xlabel='Mise à jour', ylabel='Objectif', title='Convergence')
for a in ax: a.legend()
plt.tight_layout(); plt.show()
print('Seuil SGD théorique : 2/40 = .05 ; perte pas .06 :', qloss(bad[-1]))



## Étape 4
Comparer le second moment, pas seulement la variance.


In [ ]:
# CELL 4 — Énergie des activations : q = moyenne des carrés, pas variance.
def signal_profile(kind, depth=12, width=128, seed=0):
    r = np.random.default_rng(seed)
    h = r.normal(size=(256, width))
    profile = [float(np.mean(h*h))]
    for _ in range(depth):
        std = {'small': .1/np.sqrt(width), 'xavier': 1/np.sqrt(width),
               'he': np.sqrt(2/width), 'large': 2/np.sqrt(width)}[kind]
        w = r.normal(0, std, size=(width, width))
        h = np.maximum(h @ w, 0.)
        profile.append(float(np.mean(h*h)))
    return profile

for kind in ['small', 'xavier', 'he', 'large']:
    values = np.array([signal_profile(kind, seed=s) for s in range(3)])
    plt.semilogy(values.mean(0), label=kind)
plt.xlabel('Profondeur'); plt.ylabel('Moyenne empirique de h²')
plt.title('Initialisation, ReLU et propagation du signal'); plt.legend(); plt.show()



## Étape 5
Contrôler les gradients et les dimensions.


In [ ]:
# CELL 5 — Jeu non linéaire, séparation et prétraitement.
rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, size=(640, 2))
y = (X[:, 0]*X[:, 1] > 0).astype(float).reshape(-1, 1)
idx = rng.permutation(len(X)); tr, va = idx[:480], idx[480:]
mean, std = X[tr].mean(0), X[tr].std(0)
Xtr, Xva = (X[tr]-mean)/std, (X[va]-mean)/std
ytr, yva = y[tr], y[va]

def init(seed=0, mode='he', width=16):
    r = np.random.default_rng(seed)
    scale = {'he': np.sqrt(2/2), 'tiny': 1e-4, 'zero': 0.}[mode]
    return {'W1': r.normal(0, scale, (2, width)), 'b1': np.zeros((1, width)),
            'W2': r.normal(0, 1/np.sqrt(width), (width, 1)),
            'b2': np.zeros((1, 1))}

def forward(X, p):
    u = X @ p['W1'] + p['b1']; h = np.maximum(u, 0.)
    return h @ p['W2'] + p['b2'], (u, h)

def loss_grad(X, y, p):
    z, (u, h) = forward(X, p)
    loss = float(np.mean(np.logaddexp(0., z)-y*z))
    prob = np.exp(-np.logaddexp(0., -z))
    d = (prob-y)/len(X)
    dh = (d @ p['W2'].T) * (u > 0)
    return loss, {'W1': X.T @ dh, 'b1': dh.sum(0, keepdims=True),
                  'W2': h.T @ d, 'b2': d.sum(0, keepdims=True)}

def evaluate(X, y, p):
    z, _ = forward(X, p)
    return float(np.mean(np.logaddexp(0., z)-y*z)), float(((z >= 0)==y).mean())



## Étape 6
Fixer des budgets et apparier les graines.


In [ ]:
# CELL 6 — Budgets, lots et initialisation contrôlés.
def train(kind, lr, seed=0, epochs=120, batch=64, mode='he',
          schedule='constant', clip=None, wd=0.):
    p = init(seed, mode)
    opt = Optimizer(p, kind=kind, lr=lr, wd=wd)
    order_rng = np.random.default_rng(1000+seed)
    history, steps = [], 0
    for epoch in range(epochs):
        order = order_rng.permutation(len(Xtr))
        for start in range(0, len(order), batch):
            ids = order[start:start+batch]
            if schedule == 'cosine':
                total = epochs * int(np.ceil(len(Xtr)/batch))
                opt.lr = lr*(.05+.95*.5*(1+np.cos(np.pi*steps/max(1,total-1))))
            else: opt.lr = lr
            _, g = loss_grad(Xtr[ids], ytr[ids], p)
            norm = np.sqrt(sum(float(np.sum(v*v)) for v in g.values()))
            if not np.isfinite(norm):
                return p, history, 'nonfinite'
            if clip is not None:
                factor = min(1., clip/(norm+1e-12))
                g = {k: v*factor for k,v in g.items()}
            before = {k: v.copy() for k,v in p.items()}
            opt.step(p, g); steps += 1
            update = np.sqrt(sum(float(np.sum((p[k]-before[k])**2)) for k in p))
        tl, ta = evaluate(Xtr,ytr,p); vl, vaa = evaluate(Xva,yva,p)
        history.append(dict(epoch=epoch, steps=steps, train_loss=tl, val_loss=vl,
                            train_acc=ta, val_acc=vaa, grad_norm=norm,
                            update_norm=update, lr=opt.lr))
    return p, history, 'ok'

settings = {'sgd': .1, 'momentum': .03, 'adam': .01, 'adamw': .01}
runs = {}
for kind, lr in settings.items():
    for seed in range(3):
        # AdamW a ici un wd explicite ; ce n'est donc pas une ablation de l'optimiseur seul.
        runs[kind,seed] = train(kind,lr,seed,wd=.01 if kind=='adamw' else 0.)
for kind in settings:
    scores = [runs[kind,s][1][-1]['val_acc'] for s in range(3)]
    print(kind, 'accuracy validation moyenne ± écart-type :', np.mean(scores), np.std(scores))



## Étape 7
Isoler wd=0 avant de comparer la régularisation.


In [ ]:
# CELL 7 — Courbes, puis ablation de la décroissance.
for kind in settings:
    h = runs[kind,0][1]
    plt.plot([r['steps'] for r in h], [r['train_loss'] for r in h], label=kind)
plt.xlabel('Mises à jour'); plt.ylabel('Perte entraînement'); plt.legend(); plt.show()
# Contrôle : Adam et AdamW avec wd=0 doivent coïncider ici.
p_a, _, _ = train('adam', .01, seed=0, epochs=10)
p_w, _, _ = train('adamw', .01, seed=0, epochs=10, wd=0.)
for k in p_a: np.testing.assert_allclose(p_a[k], p_w[k], atol=1e-12, rtol=1e-12)
# Comparer ensuite wd=0 et wd=.01 dans AdamW, à tous les autres réglages fixés.



## Étape 8
Localiser la panne ; ne pas supposer que pas élevé signifie divergence.


In [ ]:
# CELL 8 — Pannes contrôlées ; une hypothèse, une intervention.
diagnostics = {}
for name, kwargs in {
    'reference': dict(lr=.1),
    'pas_trop_petit': dict(lr=1e-6),
    'initialisation_nulle': dict(lr=.1, mode='zero'),
    'pas_eleve': dict(lr=3.),
}.items():
    diagnostics[name] = train('sgd', seed=0, epochs=30, **kwargs)
    p,h,status = diagnostics[name]
    print(name, status, h[-1] if h else 'aucun relevé')
# Avec ReLU et dérivée 0 en 0, des poids de première couche tous nuls bloquent les poids.
assert np.all(diagnostics['initialisation_nulle'][0]['W1'] == 0)
# Un pas élevé ne doit pas obligatoirement diverger : interpréter la trajectoire mesurée.



## Étape 9
Prédire les effets de la décroissance sur un budget fixe.


In [ ]:
# CELL 9 — Schedules : même initialisation et même nombre de mises à jour.
constant = train('adam', .01, seed=0, schedule='constant')
cosine = train('adam', .01, seed=0, schedule='cosine')
for label, result in [('constant',constant),('cosine',cosine)]:
    _, h, _ = result
    print(label, 'pas final', h[-1]['lr'], 'perte validation', h[-1]['val_loss'])
    plt.plot([r['steps'] for r in h], [r['lr'] for r in h], label=label)
plt.xlabel('Mises à jour'); plt.ylabel('Learning rate'); plt.legend(); plt.show()



## Étape 10
Option : vérifier les modes train/eval et les axes de normalisation.


In [ ]:
# CELL 10 — Option PyTorch CPU : normalisation et mesure des gradients.
# À exécuter seulement si PyTorch est installé.
import torch
from torch import nn
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2,32), nn.LayerNorm(32), nn.ReLU(), nn.Linear(32,1))
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.)
xb = torch.tensor(Xtr[:64], dtype=torch.float32)
yb = torch.tensor(ytr[:64], dtype=torch.float32)
net.train(); optimizer.zero_grad(set_to_none=True)
loss = nn.functional.binary_cross_entropy_with_logits(net(xb), yb)
loss.backward()
preclip_norm = torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.)
optimizer.step()
net.eval()
with torch.no_grad():
    validation_loss = nn.functional.binary_cross_entropy_with_logits(
        net(torch.tensor(Xva,dtype=torch.float32)), torch.tensor(yva,dtype=torch.float32))
print('Norme avant clipping :', float(preclip_norm), 'perte validation :', float(validation_loss))
# Variante : remplacer LayerNorm(32) par BatchNorm1d(32), puis comparer train/eval.
